# **Task 3: Continuous-Time Markov Model**

## **Objective**

To model transitions between healthy, ill and dead states
using a continuous-time Markov process.

The analysis estimates transition intensities, constructs
the generator matrix, calculates transition probabilities
and evaluates expected insurance claims.

##### **Import libraries and load the datasets**

In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.linalg import expm
from pathlib import Path

# Load the insurance transition dataset
data = pd.read_csv("../data/insured_transitions.csv")

# Display the first five records
data.head()

,district_id,origin,destination,exposure_py,transition_count,initial_lives
0,D01,H,I,2212,219,1137
1,D01,H,D,2212,37,1137
2,D01,I,H,311,67,112
3,D01,I,D,311,23,112
4,D02,H,I,3269,381,931


In [22]:
#Overlook of the dataset
# Examine the dataset structure
print("Dataset shape:", data.shape)

print("\nColumn names:")
print(data.columns.tolist())

print("\nDataset information:")
data.info()

Dataset shape: (96, 6)

Column names:
['district_id', 'origin', 'destination', 'exposure_py', 'transition_count', 'initial_lives']

Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 96 entries, 0 to 95
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   district_id       96 non-null     str  
 1   origin            96 non-null     str  
 2   destination       96 non-null     str  
 3   exposure_py       96 non-null     int64
 4   transition_count  96 non-null     int64
 5   initial_lives     96 non-null     int64
dtypes: int64(3), str(3)
memory usage: 5.1 KB


In [23]:
# 1. Identify the states
print("Origin states:", data["origin"].unique())
print("Destination states:", data["destination"].unique())

# 2. Examine transition combinations
print("\nTransition combinations:")
print(
    data.groupby(["origin", "destination"])[
        ["transition_count", "exposure_py"]
    ].sum()
)

# 3. Examine district coverage
print("\nNumber of districts:", data["district_id"].nunique())

print("\nRecords per district:")
print(data["district_id"].value_counts().sort_index())

# 4. Overall totals
print("\nTotal transitions:", data["transition_count"].sum())
print("Total exposure:", data["exposure_py"].sum())

Origin states: <ArrowStringArray>
['H', 'I']
Length: 2, dtype: str
Destination states: <ArrowStringArray>
['I', 'D', 'H']
Length: 3, dtype: str

Transition combinations:
                    transition_count  exposure_py
origin destination                               
H      D                         895        49764
       I                        5152        49764
I      D                         970        14417
       H                        3533        14417

Number of districts: 24

Records per district:
district_id
D01    4
D02    4
D03    4
D04    4
D05    4
D06    4
D07    4
D08    4
D09    4
D10    4
D11    4
D12    4
D13    4
D14    4
D15    4
D16    4
D17    4
D18    4
D19    4
D20    4
D21    4
D22    4
D23    4
D24    4
Name: count, dtype: int64

Total transitions: 10550
Total exposure: 128362


##### Understand how we estimate transition intensities

In [24]:
# Calculate transition intensities

transition_summary = (
    data.groupby(["origin", "destination"], as_index=False)
    .agg(
        transition_count=("transition_count", "sum"),
        exposure_py=("exposure_py", "sum")
    )
)

transition_summary["intensity"] = (
    transition_summary["transition_count"] /
    transition_summary["exposure_py"]
)

transition_summary

,origin,destination,transition_count,exposure_py,intensity
0,H,D,895,49764,0.017985
1,H,I,5152,49764,0.103529
2,I,D,970,14417,0.067282
3,I,H,3533,14417,0.245058


##### **Construction of the Generator matrix Q**

In [25]:
# Define the transition intensities

q_HI = 0.103529
q_HD = 0.017985

q_IH = 0.245058
q_ID = 0.067282

# Construct the generator matrix
Q = np.array([
    [-(q_HI + q_HD), q_HI, q_HD],
    [q_IH, -(q_IH + q_ID), q_ID],
    [0, 0, 0]
])

# Display the generator matrix
print("Generator Matrix Q:")
print(Q)

# Verify that every row sums to zero
print("\nRow sums:")
print(Q.sum(axis=1))

Generator Matrix Q:
[[-0.121514  0.103529  0.017985]
 [ 0.245058 -0.31234   0.067282]
 [ 0.        0.        0.      ]]

Row sums:
[ 0.00000000e+00 -1.38777878e-17  0.00000000e+00]


## **Comment**

        H          I          D
H   -0.121514   0.103529   0.017985
I    0.245058  -0.312340   0.067282
D    0          0          0

Transition	Intensity	Interpretation
H → I	0.103529	Healthy individuals become ill at a rate of 0.103529 per person-year.

H → D	0.017985	Healthy individuals die at a rate of 0.017985 per person-year.

I → H	0.245058	Ill individuals recover at a rate of 0.245058 per person-year.

I → D	0.067282	Ill individuals die at a rate of 0.067282 per person-year.

## **Part B :Calculating the mean holding time**
b. Calculate the mean holding time in each living state and the next-jump destinations and probabilities
from the ill state. Explain why the probability of eventual recovery on the next jump differs from the
probability of being healthy at the end of one year.

##### **Mean Holding Time**

In [26]:
# Calculate expected holding times

holding_H = 1 / (-Q[0, 0])
holding_I = 1 / (-Q[1, 1])

print(f"Expected holding time in Healthy: {holding_H:.2f} years")
print(f"Expected holding time in Ill: {holding_I:.2f} years")
print("Expected holding time in Dead: Infinite")

Expected holding time in Healthy: 8.23 years
Expected holding time in Ill: 3.20 years
Expected holding time in Dead: Infinite


##### **probabilities of the next transition**

In [27]:
# Next-transition probabilities

p_HI = Q[0, 1] / -Q[0, 0]
p_HD = Q[0, 2] / -Q[0, 0]

p_IH = Q[1, 0] / -Q[1, 1]
p_ID = Q[1, 2] / -Q[1, 1]

print("From Healthy:")
print(f"Probability of moving to Ill: {p_HI:.4f}")
print(f"Probability of moving to Dead: {p_HD:.4f}")

print("\nFrom Ill:")
print(f"Probability of recovering to Healthy: {p_IH:.4f}")
print(f"Probability of moving to Dead: {p_ID:.4f}")

From Healthy:
Probability of moving to Ill: 0.8520
Probability of moving to Dead: 0.1480

From Ill:
Probability of recovering to Healthy: 0.7846
Probability of moving to Dead: 0.2154


##### **The Transition Probability Matrix P(t)**

In [28]:
# Calculate transition probability matrices

P1 = expm(Q * 1)
P2 = expm(Q * 2)


print("Transition Probability Matrix After 1 Year:")
print(np.round(P1, 4))

print("\nTransition Probability Matrix After 2 Years:")
print(np.round(P2, 4))



Transition Probability Matrix After 1 Year:
[[0.8962 0.0838 0.02  ]
 [0.1984 0.7417 0.0599]
 [0.     0.     1.    ]]

Transition Probability Matrix After 2 Years:
[[0.8197 0.1373 0.043 ]
 [0.3249 0.5667 0.1084]
 [0.     0.     1.    ]]


#### **Verify the transition probability matrices**
##### ***A valid transition probability matrix must satisfy three conditions:***
1.Check that probabilities are
2.nonnegative, rows sum to one, death is absorbing and P(2) agrees with P(1)2
 within numerical tolerance.
3.Multiply the pooled initial count row vector by each matrix to obtain expected occupancy.

In [29]:
# Validate P(1) and P(2)

for name, P in [("P(1)", P1), ("P(2)", P2)]:

    print(f"\n{name}")

    # Check nonnegative probabilities
    print("Nonnegative probabilities:",
          np.all(P >= -1e-12))

    # Check row sums equal one
    print("Rows sum to one:",
          np.allclose(P.sum(axis=1), 1))

    # Check death is absorbing
    print("Death is absorbing:",
          np.allclose(P[2], [0, 0, 1]))


# Check Chapman-Kolmogorov relationship

print("\nChapman-Kolmogorov check:")

print("P(2) agrees with P(1) squared:",
      np.allclose(P2, P1 @ P1))

print("Maximum numerical difference:",
      np.max(np.abs(P2 - P1 @ P1)))


P(1)
Nonnegative probabilities: True
Rows sum to one: True
Death is absorbing: True

P(2)
Nonnegative probabilities: True
Rows sum to one: True
Death is absorbing: True

Chapman-Kolmogorov check:
P(2) agrees with P(1) squared: True
Maximum numerical difference: 1.1102230246251565e-16


Construct the pooled initial count row vector

In [30]:
# Extract one initial count per district and origin state

initial_counts = (
    data[["district_id", "origin", "initial_lives"]]
    .drop_duplicates(subset=["district_id", "origin"])
)

# Pool initial counts across all districts
pooled_counts = (
    initial_counts.groupby("origin")["initial_lives"]
    .sum()
)

# Arrange counts in H, I, D order
initial_vector = np.array([
    pooled_counts.get("H", 0),
    pooled_counts.get("I", 0),
    0
])

print("Pooled initial population:")
print("Healthy:", initial_vector[0])
print("Ill:", initial_vector[1])
print("Dead:", initial_vector[2])

print("\nInitial count row vector:")
print(initial_vector)

print("\nTotal initial population:", initial_vector.sum())

Pooled initial population:
Healthy: 20080
Ill: 3576
Dead: 0

Initial count row vector:
[20080  3576     0]

Total initial population: 23656


expected state occupancy

In [31]:
# Calculate expected occupancy after 1 and 2 years

occupancy_1 = initial_vector @ P1
occupancy_2 = initial_vector @ P2

# Display results
occupancy_results = pd.DataFrame({
    "State": ["Healthy", "Ill", "Dead"],
    "Initial population": initial_vector,
    "After 1 year": occupancy_1,
    "After 2 years": occupancy_2
})

print(occupancy_results.round(2).to_string(index=False))

print("\nTotal population after 1 year:",
      round(occupancy_1.sum(), 2))

print("Total population after 2 years:",
      round(occupancy_2.sum(), 2))

  State  Initial population  After 1 year  After 2 years
Healthy               20080      18704.24       17621.99
    Ill                3576       4335.25        4783.04
   Dead                   0        616.50        1250.96

Total population after 1 year: 23656.0
Total population after 2 years: 23656.0


##### **Comment**
Healthy population: Decreases from 20,080 to approximately 17,622 after two years. This reflects individuals becoming ill or dying.

Ill population: Increases from 3,576 to approximately 4,783. Although individuals recover, new cases from the Healthy state contribute to the increase.

Deaths: After two years, approximately 1,251 individuals are expected to be in the Dead state.

The model therefore suggests increasing mortality accumulation over time.


## **Part3:Calculation of benefit payments**
Assume each covered death triggers USD 1,000, all survivors remain covered, and ignore discounting.
Estimate benefit payments in year one and year two separately. Explain one limitation of pooling constant
intensities

In [32]:
# Calculate deaths occurring in each year

deaths_year1 = occupancy_1[2] - initial_vector[2]

deaths_year2 = occupancy_2[2] - occupancy_1[2]

# Benefit payment per death
death_benefit = 1000

# Calculate annual benefit payments
benefit_year1 = deaths_year1 * death_benefit
benefit_year2 = deaths_year2 * death_benefit

# Display results
print(f"Expected deaths in Year 1: {deaths_year1:.2f}")
print(f"Expected deaths in Year 2: {deaths_year2:.2f}")

print(f"\nBenefit payments in Year 1: ${benefit_year1:,.2f}")
print(f"Benefit payments in Year 2: ${benefit_year2:,.2f}")

print(f"\nTotal two-year benefit payments: "
      f"${benefit_year1 + benefit_year2:,.2f}")

Expected deaths in Year 1: 616.50
Expected deaths in Year 2: 634.46

Benefit payments in Year 1: $616,504.14
Benefit payments in Year 2: $634,460.17

Total two-year benefit payments: $1,250,964.31


In [33]:
benefits =pd.DataFrame({"year":[1,2],"Payment":[benefit_year1,benefit_year2]})
benefits

,year,Payment
0,1,616504.135624
1,2,634460.169523


## **Interpretation and explanation**

One important limitation is that pooling assumes the same transition intensities apply to all individuals across the 24 districts and remain constant over time.

In reality, mortality and illness rates may differ considerably between districts due to differences in age structure, healthcare access, socioeconomic conditions and other risk factors.

For example, a district with an older population may experience higher mortality than a district with a younger population.

Pooling these districts can hide such differences and produce estimates that do not accurately represent individual districts or particular risk groups

**interpretation**
The model estimates that the insurer will pay approximately $616,500 in Year 1 and $634,460 in Year 2, assuming a fixed death benefit of $1,000 and no discounting. The increase in annual payments reflects the growing expected number of deaths in the covered population.

In [34]:
occupancy_results.to_csv('../outputs/markov/occupancy_results.csv')
benefits.to_csv('../outputs/markov/insurance_benefits.csv')